In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

def load_and_engineer_data(ticker="RELIANCE.NS", start_date="2020-01-01", end_date="2026-01-01"):
    # 1. Fetch Data
    df = yf.download(ticker, start=start_date, end=end_date)
    
    # 2. Feature Engineering (The inputs for your model)
    # Moving Averages (Trend)
    df['SMA_20'] = df['Close'].rolling(window=20).mean()
    df['SMA_50'] = df['Close'].rolling(window=50).mean()
    
    # Volatility & Momentum
    df['Daily_Return'] = df['Close'].pct_change()
    df['Volatility_20d'] = df['Daily_Return'].rolling(window=20).std()
    
    # 3. Define the Target Variable (What we want to predict)
    # We want to predict the NEXT day's closing price based on TODAY'S features
    df['Target_Next_Close'] = df['Close'].shift(-1)  
    
    # Drop rows with NaN values created by rolling windows and shifting
    df.dropna(inplace=True)
    
    return df

data = load_and_engineer_data()

[*********************100%***********************]  1 of 1 completed


In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import xgboost as xgb
import math

# Define Features (X) and Target (y)
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_20', 'SMA_50', 'Volatility_20d']
X = data[features]
y = data['Target_Next_Close']

# Chronological Split: 80% for training, 20% for testing
split_idx = int(len(data) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training records: {len(X_train)}, Testing records: {len(X_test)}")

Training records: 1149, Testing records: 288


In [3]:
# Initialize and Train the Model
model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# Make Predictions on the Test Set
predictions = model.predict(X_test)

# Evaluate the Model
rmse = math.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)

print(f"Model RMSE: ₹{rmse:.2f}")
print(f"Model MAE: ₹{mae:.2f}")

Model RMSE: ₹22.84
Model MAE: ₹18.20


In [4]:
import streamlit as st
import plotly.graph_objects as go

st.title("Indian Stock Market Predictor 📈")

# User Inputs
ticker = st.sidebar.text_input("Enter Ticker (e.g., RELIANCE.NS, TCS.NS)", "RELIANCE.NS")
if st.sidebar.button("Train Model & Predict"):
    
    with st.spinner('Fetching data and training model...'):
        # 1. Run your functions from above
        df = load_and_engineer_data(ticker=ticker)
        # ... (Run the train/test split and modeling code here) ...
        
        st.success(f"Model trained successfully! RMSE: ₹{rmse:.2f}")
        
        # 2. Visualize Actual vs Predicted using Plotly
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=y_test.index, y=y_test.values, mode='lines', name='Actual Price'))
        fig.add_trace(go.Scatter(x=y_test.index, y=predictions, mode='lines', name='Predicted Price'))
        fig.update_layout(title=f"{ticker} - Actual vs Predicted", xaxis_title="Date", yaxis_title="Price (₹)")
        
        st.plotly_chart(fig)

2026-03-29 11:16:20.143 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 11:16:20.270 
  command:

    streamlit run C:\Users\muhda\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-29 11:16:20.271 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 11:16:20.271 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 11:16:20.272 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 11:16:20.272 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 11:16:20.273 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 11:16:20.274 Session state does not 